### Step 1: Install and Import Necessary Libraries

In [2]:
!pip install sentence-transformers

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

### Step 2: Load the Sentence Embedding Model

In [3]:
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


### Step 3: Define Documents and Sample Query

In [4]:
documents = [
    "Student can apply for scolarships and finincial aid.",
    "Python is a popular programming language used in AI.",
    "The university library is open from 9AM to 8PM.",
    "Students can complete there internship during there studies.",
    "Machine learning allows computers to learn from data.",
    "The university offers courses in pyhton programming, machine Learning, and artificial intelligence.",
    "Students must complete all required examinations and credits to graduate.",
    "the University Provides career counseling and internship guidance for students."
]

query = "Is Python a Programming Language?"
print("User Question:", query)

User Question: Is Python a Programming Language?


### Step 4: Compute Document Embeddings

In [5]:
document_embeddings = model.encode(documents)
print("Document Embeddings Shape:", document_embeddings.shape)

Document Embeddings Shape: (8, 384)


### Step 5: Load Generative Language Model (FLAN-T5-Small)

In [6]:
print("Loading language model...")
model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
generator = AutoModelForSeq2SeqLM.from_pretrained(model_name)
print("Generative model loaded successfully!")

Loading language model...


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Generative model loaded successfully!


### Step 6: Define Document Retrieval and Prompt Building Functions

In [7]:
def retrieve_documents(query, top_k=2):
    query_embedding = model.encode([query])
    similarities = cosine_similarity(query_embedding, document_embeddings)[0]
    ranked_indices = np.argsort(similarities)[::-1]

    retrieved_documents = []
    for index in ranked_indices[:top_k]:
        retrieved_documents.append({
            "text": documents[index],
            "score": float(similarities[index])
        })
    return retrieved_documents

def create_prompt(query, retrieved_results):
    context = "\n".join(item["text"] for item in retrieved_results)

    prompt = f"""Answer the following question using the context below.
If the context doesn't have the answer, say "I don't have enough info to answer."

Context:
{context}

Question:
{query}

Answer:
"""
    return prompt

### Step 7: Retrieve Documents for Our Query

In [8]:
retrieved_docs = retrieve_documents(query, top_k=2)
print("Retrieved Documents:")
for i, item in enumerate(retrieved_docs):
    print(f"{i+1}. {item['text']} (Similarity Score: {item['score']:.4f})")

Retrieved Documents:
1. Python is a popular programming language used in AI. (Similarity Score: 0.7557)
2. The university offers courses in pyhton programming, machine Learning, and artificial intelligence. (Similarity Score: 0.4664)


### Step 8: Build Prompt and Generate Answer

In [9]:
prompt = create_prompt(query, retrieved_docs)
print("Generated Prompt:\n", prompt)

inputs = tokenizer(prompt, return_tensors="pt")
outputs = generator.generate(**inputs, max_new_tokens=150)
result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Model Output:", result)

Generated Prompt:
 Answer the following question using the context below.
If the context doesn't have the answer, say "I don't have enough info to answer."

Context:
Python is a popular programming language used in AI.
The university offers courses in pyhton programming, machine Learning, and artificial intelligence.

Question:
Is Python a Programming Language?

Answer:

Model Output: Yes


### Step 9: Define the RAG Chatbot Loop function

In [10]:
def rag_chatbot(query_input):
    docs = retrieve_documents(query_input, top_k=2)
    prompt_str = create_prompt(query_input, docs)
    inputs_t = tokenizer(prompt_str, return_tensors="pt")
    outputs_t = generator.generate(**inputs_t, max_new_tokens=150)
    ans = tokenizer.decode(outputs_t[0], skip_special_tokens=True)
    print("\n--- RAG Response ---")
    print(ans)
    print("---------------------\n")

### Step 10: Interactive Chat Loop (Optional Execution)

In [11]:
while True:
    user_question = input("Enter your question (or type 'exit' to quit): ")
    if not user_question.strip():
        print("Please enter a valid question")
        continue
    if user_question.strip().lower() == 'exit':
        print("Goodbye!")
        break

    rag_chatbot(user_question)

Enter your question (or type 'exit' to quit): hii

--- RAG Response ---
I don't have enough info to answer
---------------------

Enter your question (or type 'exit' to quit): is python a language'

--- RAG Response ---
yes
---------------------

Enter your question (or type 'exit' to quit): exit
Goodbye!


In [19]:
!pip install pypdf

import os
from pypdf import PdfReader

pdf_files = [
    "/content/Guidelines-College-SPOC.pdf",
    "/content/NeuraForge_SIH26024_Idea_Submission.pdf"
]

new_documents = []

for pdf_path in pdf_files:
    if os.path.exists(pdf_path):
        print(f"Extracting text from: {pdf_path}")
        reader = PdfReader(pdf_path)
        for page_num, page in enumerate(reader.pages):
            text = page.extract_text()
            if text and text.strip():
                paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]
                for para in paragraphs:
                    if len(para) > 30:
                        new_documents.append(f"[{os.path.basename(pdf_path)} - Page {page_num+1}] {para}")
    else:
        print(f"File not found: {pdf_path}")

print(f"Successfully parsed {len(new_documents)} chunks from PDFs.")


documents = new_documents
document_embeddings = model.encode(documents)
print("Document Embeddings shape updated to:", document_embeddings.shape)

Extracting text from: /content/Guidelines-College-SPOC.pdf
Extracting text from: /content/NeuraForge_SIH26024_Idea_Submission.pdf
Successfully parsed 31 chunks from PDFs.
Document Embeddings shape updated to: (31, 384)


In [ ]:

if 'new_documents' in globals():
    documents = new_documents
    document_embeddings = model.encode(documents)


In [ ]:
while True:
    user_question = input("Enter your question (or type 'exit' to quit): ")
    if not user_question.strip():
        print("Please enter a valid question")
        continue
    if user_question.strip().lower() == 'exit':
        print("Goodbye!")
        break

    rag_chatbot(user_question)

Enter your question (or type 'exit' to quit): what is in the context 

--- RAG Response ---
I don't have the answer
---------------------

Enter your question (or type 'exit' to quit): what is spoc

--- RAG Response ---
College SPOC
---------------------

